In [8]:
# ============================================================
# CELL 1 — Spark Configuration + Imports
# GlobalWatch: Bronze Ingestion — OpenAQ API
# ============================================================

# --- Spark Optimization Settings ---
# AQE: lets Spark dynamically optimize shuffle partitions at runtime
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Coalesce small partitions after shuffle — avoids 200 tiny files
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# Auto-detect and handle skewed partitions (e.g. high-volume city stations)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

# --- Standard Imports ---
import requests                              # HTTP calls to OpenAQ API
from datetime import datetime, timezone, timedelta   # UTC timestamps + freshness window
from pyspark.sql import functions as F       # PySpark column functions
from pyspark.sql.types import *              # Schema type definitions

# --- Database Context ---
# Fabric encodes the lakehouse path into an internal DB name
# We let Fabric tell us its own name rather than hardcoding it
# This avoids the SCHEMA_NOT_FOUND error from name duplication
DB = spark.sql("SELECT current_database()").collect()[0][0]

# --- API Key ---
# Stored securely in Fabric Environment Spark properties
# Key: spark.openaq.api.key — never hardcoded in notebook
OPENAQ_API_KEY = spark.conf.get("spark.openaq.api.key")

print(f"Config loaded ✅")
print(f"DB context: {DB}")
print(f"API Key loaded: {'✅' if OPENAQ_API_KEY else '❌ NOT FOUND — check environment'}")

StatementMeta(, 62002255-83f5-40c9-900e-4f4320c075a0, 12, Finished, Available, Finished, False)

Config loaded ✅
DB context: chimcobldhq2aprcdth62r3nc5q66q1dchinc9b2e9nmsuj5btjmorr2c5m7eobkcdk2ap32ds
API Key loaded: ✅


In [9]:
# ============================================================
# CELL 2 — Watermark Control Table
# Purpose: Track last loaded date per source
# Pattern: Incremental ingestion — only pull new data each run
# ============================================================

# Create watermark table if it doesn't exist
# USING DELTA: enables ACID transactions + time travel
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB}.watermark_control (
    source_name      STRING,      -- source identifier e.g. 'openaq_batch'
    last_loaded_date DATE,        -- date of last successful load
    last_loaded_ts   TIMESTAMP    -- full timestamp of last successful load
) USING DELTA
""")

# Seed initial watermark only if table is empty
count = spark.sql(f"""
    SELECT COUNT(*) as cnt FROM {DB}.watermark_control
""").collect()[0]['cnt']

if count == 0:
    # Start from Jan 2024 — historical backfill starting point
    spark.sql(f"""
    INSERT INTO {DB}.watermark_control VALUES
    ('openaq_batch', '2024-01-01', '2024-01-01T00:00:00'),
    ('waqi_batch',   '2024-01-01', '2024-01-01T00:00:00')
    """)
    print("Initial watermark seeded ✅")
else:
    print(f"Watermark table already has {count} rows — skipping seed")

# Display current watermark state
spark.sql(f"SELECT * FROM {DB}.watermark_control").show()

# --- Bind the watermark into the session ---
# This read is what makes the table load-bearing. Without it the ingest
# cell falls through to its fallback and every run reloads everything,
# which is how readings dated 2016 reached Gold.
_wm = spark.sql(f"""
    SELECT last_loaded_ts
    FROM {DB}.watermark_control
    WHERE source_name = 'openaq_batch'
""").collect()

if not _wm or _wm[0]["last_loaded_ts"] is None:
    raise RuntimeError(
        "No watermark row for 'openaq_batch'. Refusing to run: a missing "
        "watermark previously defaulted to 2000-01-01 and silently loaded "
        "the full history."
    )

last_watermark = _wm[0]["last_loaded_ts"]
print(f"Watermark in effect: {last_watermark}")
print("Watermark table ready ✅")

StatementMeta(, 62002255-83f5-40c9-900e-4f4320c075a0, 13, Finished, Available, Finished, False)

Watermark table already has 2 rows — skipping seed
+------------+----------------+--------------------+
| source_name|last_loaded_date|      last_loaded_ts|
+------------+----------------+--------------------+
|openaq_batch|      2026-08-09|2026-08-09 11:53:...|
|  waqi_batch|      2024-01-01| 2024-01-01 00:00:00|
+------------+----------------+--------------------+

Watermark table ready ✅


In [10]:
# ============================================================
# CELL 3 — OpenAQ API Functions + Connectivity Test
# API: OpenAQ v3 — https://api.openaq.org/v3
# Auth: X-API-Key header (key from Fabric environment)
# Free tier: 60 req/min, global coverage, 10K+ stations
# ============================================================

OPENAQ_BASE = "https://api.openaq.org/v3"

def fetch_openaq_locations(limit=50, page=1):
    """
    Fetch air quality station metadata.
    Returns station ID, name, country, coordinates.
    Used to discover which stations to pull measurements from.
    """
    url = f"{OPENAQ_BASE}/locations"
    params = {
        "limit": limit,   # stations per page (max 1000)
        "page": page      # pagination — 1-indexed
    }
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY   # required for v3
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        if r.status_code == 200:
            return r.json()
        else:
            print(f"  API error {r.status_code}: {r.text[:100]}")
            return None
    except Exception as e:
        print(f"  Request failed: {e}")
        return None

def fetch_openaq_measurements(location_id, limit=20):
    """
    Fetch latest pollutant measurements for a specific station.
    Returns PM2.5, PM10, NO2, CO, O3 readings with timestamps.
    """
    url = f"{OPENAQ_BASE}/locations/{location_id}/measurements"
    params = {"limit": limit}
    headers = {
        "Accept": "application/json",
        "X-API-Key": OPENAQ_API_KEY
    }
    try:
        r = requests.get(url, params=params, headers=headers, timeout=30)
        if r.status_code == 200:
            return r.json()
        return None
    except Exception as e:
        print(f"  Measurement fetch failed for location {location_id}: {e}")
        return None

# --- Connectivity Test ---
print("Testing OpenAQ API connectivity...")
test = fetch_openaq_locations(limit=3)
if test:
    total_found = test['meta']['found']
    print(f"API connected ✅ — {total_found} stations globally")
    print("Sample stations:")
    for loc in test['results']:
        code = loc.get('country', {}).get('code', '??')
        name = loc.get('name', 'N/A')
        country = loc.get('country', {}).get('name', 'N/A')
        print(f"  → [{code}] {name} | {country}")
else:
    print("❌ API connection failed — check API key in environment settings")

StatementMeta(, 62002255-83f5-40c9-900e-4f4320c075a0, 14, Finished, Available, Finished, False)

Testing OpenAQ API connectivity...
API connected ✅ — >3 stations globally
Sample stations:
  → [GH] NMA - Nima | Ghana
  → [GH] NMT - Nima | Ghana
  → [GH] JTA - Jamestown | Ghana


In [12]:
# ============================================================
# CELL 4 — Optimized: Parallel fetch across 30 countries
# Changes vs original:
#   1. Target 70 specific countries via API filter
#   2. ThreadPoolExecutor (10 workers) for parallel calls
#   3. /latest endpoint — 1 call per sensor vs 20
#   4. Rate limit: 60 req/min respected via semaphore
# ============================================================

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Semaphore

# 70 target countries — spread across all continents
TARGET_COUNTRIES = [
    # Asia (25)
    "IN", "CN", "JP", "ID", "PK", "BD", "PH", "VN", "TH", "MN",
    "KZ", "UZ", "KR", "TR", "SA", "IL", "KW", "QA", "AE", "SG",
    "MY", "LK", "NP", "MM", "KH",
    # Europe (20)
    "GB", "DE", "FR", "PL", "NL", "ES", "IT", "UA", "RU", "BE",
    "CH", "SE", "NO", "CZ", "RO", "PT", "GR", "HU", "AT", "FI",
    # Americas (15)
    "US", "BR", "MX", "CA", "AR", "CO", "PE", "CL", "EC", "BO",
    "VE", "PY", "UY", "CR", "PA",
    # Africa (10)
    "ZA", "NG", "KE", "ET", "GH", "EG", "MA", "TZ", "UG", "SN",
]

schema = StructType([
    StructField("location_id",   IntegerType()),
    StructField("location_name", StringType()),
    StructField("city",          StringType()),
    StructField("country_code",  StringType()),
    StructField("country_name",  StringType()),
    StructField("latitude",      DoubleType()),
    StructField("longitude",     DoubleType()),
    StructField("parameter",     StringType()),
    StructField("value",         DoubleType()),
    StructField("unit",          StringType()),
    StructField("reading_ts",    TimestampType()),
    StructField("ingestion_ts",  TimestampType()),
    StructField("source_system", StringType())
])

# Rate limiter — max 10 concurrent requests
SEMAPHORE = Semaphore(10)

def fetch_sensor_latest_v2(sensor_id):
    """
    Fetch single latest reading for a sensor via /latest endpoint.
    3-5x fewer API calls vs /measurements?limit=20.
    """
    with SEMAPHORE:
        url = f"{OPENAQ_BASE}/sensors/{sensor_id}/measurements"
        params = {"limit": 1}
        headers = {
            "Accept": "application/json",
            "X-API-Key": OPENAQ_API_KEY
        }
        try:
            r = requests.get(url, params=params, headers=headers, timeout=15)
            if r.status_code == 200:
                results = r.json().get("results", [])
                return results[0] if results else None
        except:
            pass
        return None

def process_location(loc):
    """
    Process one location — fetch all sensor latest readings.
    Returns list of row dicts for Bronze table.
    """
    rows = []
    now = datetime.utcnow()

    location_id   = loc.get("id")
    location_name = loc.get("name", "")
    city          = loc.get("locality") or loc.get("country", {}).get("name", "")
    country       = loc.get("country", {}) if isinstance(loc.get("country"), dict) else {}
    country_code  = country.get("code", "")
    country_name  = country.get("name", "")
    coords        = loc.get("coordinates", {}) if isinstance(loc.get("coordinates"), dict) else {}
    latitude      = coords.get("latitude")
    longitude     = coords.get("longitude")
    sensors       = loc.get("sensors", [])

    if not sensors:
        return rows

    # Fetch latest reading for each sensor in parallel
    sensor_futures = {}
    with ThreadPoolExecutor(max_workers=5) as inner_exec:
        for sensor in sensors:
            sid = sensor.get("id")
            param = sensor.get("parameter", {})
            param_name = param.get("name", "") if isinstance(param, dict) else ""
            param_unit = param.get("units", "") if isinstance(param, dict) else ""

            if not sid or not param_name:
                continue

            future = inner_exec.submit(fetch_sensor_latest_v2, sid)
            sensor_futures[future] = (param_name, param_unit)

        for future, (param_name, param_unit) in sensor_futures.items():
            try:
                measurement = future.result(timeout=20)
                if not measurement:
                    continue

                value = measurement.get("value")
                period = measurement.get("period", {})
                date_from = period.get("datetimefrom", {}) if isinstance(period, dict) else {}
                ts_str = date_from.get("utc") if isinstance(date_from, dict) else None

                if value is None or not isinstance(value, (int, float)):
                    continue

                try:
                    reading_ts = datetime.strptime(ts_str[:19], "%Y-%m-%dT%H:%M:%S") if ts_str else now
                except:
                    reading_ts = now

                rows.append({
                    "location_id":   location_id,
                    "location_name": location_name,
                    "city":          city,
                    "country_code":  country_code,
                    "country_name":  country_name,
                    "latitude":      float(latitude) if latitude else None,
                    "longitude":     float(longitude) if longitude else None,
                    "parameter":     param_name,
                    "value":         float(value),
                    "unit":          param_unit,
                    "reading_ts":    reading_ts,
                    "ingestion_ts":  now,
                    "source_system": "openaq_v3"
                })
            except Exception:
                continue

    return rows

# ── Main ingestion loop ────────────────────────────────────
all_rows = []
processed = skipped = errors = 0
start_time = time.time()

print(f"Fetching stations for {len(TARGET_COUNTRIES)} countries...")
print(f"Target countries: {', '.join(TARGET_COUNTRIES)}")
print("-" * 60)

# Fetch locations per country (up to ~40 stations per country = ~800 total)
all_locations = []
for country_code in TARGET_COUNTRIES:
    try:
        url = f"{OPENAQ_BASE}/locations"
        params = {
            "limit": 40,
            "iso": country_code
        }
        headers = {
            "Accept": "application/json",
            "X-API-Key": OPENAQ_API_KEY
        }
        r = requests.get(url, params=params, headers=headers, timeout=20)
        if r.status_code == 200:
            locs = r.json().get("results", [])
            all_locations.extend(locs)
            print(f"  {country_code}: {len(locs)} stations found")
        else:
            print(f"  {country_code}: API error {r.status_code}")
    except Exception as e:
        print(f"  {country_code}: {e}")

print(f"\nTotal stations to process: {len(all_locations)}")
print("-" * 60)

# Process all locations in parallel (10 workers)
print("Fetching measurements in parallel...")
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(process_location, loc): loc for loc in all_locations}
    completed = 0
    for future in as_completed(futures):
        try:
            rows = future.result()
            all_rows.extend(rows)
            processed += 1
        except Exception:
            errors += 1
        completed += 1
        if completed % 50 == 0:
            elapsed = time.time() - start_time
            print(f"  Progress: {completed}/{len(all_locations)} stations | {len(all_rows)} readings | {elapsed:.0f}s")

elapsed = time.time() - start_time
print(f"\n✅ Done in {elapsed:.0f}s")
print(f"   Stations processed : {processed}")
print(f"   Errors             : {errors}")
print(f"   Total readings     : {len(all_rows)}")

# ── Write to Bronze Delta ──────────────────────────────────
if all_rows:
    from pyspark.sql import Row
    from datetime import datetime

    rows_rdd = spark.sparkContext.parallelize(all_rows)
    new_df = spark.createDataFrame(rows_rdd, schema=schema) \
        .withColumn("ingestion_date", F.to_date("ingestion_ts"))


    # Fail loudly rather than defaulting. A watermark that silently falls
    # back to 2000-01-01 disables the filter without failing the run.
    if "last_watermark" not in globals():
        raise RuntimeError(
            "last_watermark is not set — run the watermark control cell first."
        )

    # --- Staleness cutoff (not a high-water mark) ---
    # /sensors/{id}/measurements?limit=1 returns each sensor's latest reading,
    # so recency varies per sensor: dead stations still answer with values
    # years old. A single global high-water mark is the wrong tool here — once
    # set to the newest sensor's timestamp it permanently excludes every
    # slower-reporting station, including healthy ones lagging a few hours.
    # Bound on absolute age instead, which drops dead stations only.
    STALE_AFTER_DAYS = 7
    cutoff = datetime.now(timezone.utc).replace(tzinfo=None) \
             - timedelta(days=STALE_AFTER_DAYS)

    fetched = new_df.count()
    new_df = new_df.filter(F.col("reading_ts") >= F.lit(cutoff)).cache()
    new_count = new_df.count()

    print(f"\n   Fetched            : {fetched}")
    print(f"   Cutoff             : {cutoff} ({STALE_AFTER_DAYS}d)")
    print(f"   Dropped as stale   : {fetched - new_count}")
    print(f"   To load            : {new_count}")

    if new_count > 0:
        # --- Idempotent upsert on the natural key ---
        # mode("append") re-landed every row on each run. MERGE on the grain
        # Silver deduplicates by makes re-runs and backfills safe.
        from delta.tables import DeltaTable

        target = f"{DB}.raw_openaq_readings"

        if not spark.catalog.tableExists(target):
            new_df.write \
                .format("delta") \
                .partitionBy("ingestion_date") \
                .option("mergeSchema", "true") \
                .saveAsTable(target)
            print(f"✅ Created Bronze with {new_count} rows")
        else:
            before = spark.table(target).count()
            DeltaTable.forName(spark, target).alias("t").merge(
                new_df.alias("s"),
                "t.location_id = s.location_id "
                "AND t.parameter   = s.parameter "
                "AND t.reading_ts  = s.reading_ts"
            ).whenNotMatchedInsertAll().execute()
            after = spark.table(target).count()
            print(f"✅ Merged into Bronze: {after - before} new, "
                  f"{new_count - (after - before)} already present")

        # Hand the loaded batch to the validation cell so the new watermark
        # can be derived from the data rather than from the clock.
        batch_max_reading_ts = new_df.agg(F.max("reading_ts")).collect()[0][0]
    else:
        batch_max_reading_ts = None
        print("⚠️ Nothing within the freshness window — nothing written")
else:
    print("❌ No data collected")

StatementMeta(, 62002255-83f5-40c9-900e-4f4320c075a0, 16, Finished, Available, Finished, False)

Fetching stations for 30 countries...
Target countries: IN, CN, US, GB, DE, FR, JP, BR, AU, ZA, NG, MX, ID, PK, BD, PL, NL, TH, CL, MN, KE, ET, PH, VN, TR, SA, EG, AR, CO, GH
------------------------------------------------------------
  IN: 17 stations found
  CN: 17 stations found
  US: 17 stations found
  GB: 17 stations found
  DE: 17 stations found
  FR: 17 stations found
  JP: 17 stations found
  BR: 17 stations found
  AU: 17 stations found
  ZA: 17 stations found
  NG: 17 stations found
  MX: 17 stations found
  ID: 17 stations found
  PK: 17 stations found
  BD: 17 stations found
  PL: 17 stations found
  NL: 17 stations found
  TH: 17 stations found
  CL: 17 stations found
  MN: 17 stations found
  KE: 17 stations found
  ET: 9 stations found
  PH: 17 stations found
  VN: 17 stations found
  TR: 17 stations found
  SA: 9 stations found
  EG: 2 stations found
  AR: 17 stations found
  CO: 17 stations found
  GH: 17 stations found

Total stations to process: 479
---------------

In [13]:
# ============================================================
# CELL 5 — Validation + Watermark Update
# ============================================================

# --- Row Count ---
total = spark.sql(f"""
    SELECT COUNT(*) as total FROM {DB}.raw_openaq_readings
""").collect()[0]['total']
print(f"Total rows in Bronze: {total}")
assert total > 0, "❌ VALIDATION FAILED: Bronze table is empty!"

# --- Data Quality Summary ---
print("\nTop 10 by reading count:")
spark.sql(f"""
    SELECT
        country_code,
        country_name,
        parameter,
        ROUND(AVG(value), 2)  AS avg_value,
        ROUND(MIN(value), 2)  AS min_value,
        ROUND(MAX(value), 2)  AS max_value,
        COUNT(*)              AS readings
    FROM {DB}.raw_openaq_readings
    GROUP BY country_code, country_name, parameter
    ORDER BY readings DESC
    LIMIT 10
""").show(truncate=False)

# --- Partition Check ---
print("Partitions written:")
spark.sql(f"""
    SELECT ingestion_date, COUNT(*) as rows
    FROM {DB}.raw_openaq_readings
    GROUP BY ingestion_date
""").show()

# --- Update Watermark ---
# Derive it from the data, never from the clock. reading_ts always trails
# run time, so a current_timestamp() watermark would exclude every row on
# the following run and the pipeline would go quietly empty.
_batch_max = globals().get("batch_max_reading_ts")

if _batch_max is None:
    print("No rows loaded this run — watermark left unchanged")
else:
    spark.sql(f"""
        UPDATE {DB}.watermark_control
        SET last_loaded_ts   = TIMESTAMP'{_batch_max}',
            last_loaded_date = DATE'{_batch_max.date()}'
        WHERE source_name = 'openaq_batch'
    """)
    print(f"Watermark advanced to {_batch_max} ✅")
print("Bronze ingestion complete ✅")
print("Next: Run 04_silver_transform notebook")

StatementMeta(, 62002255-83f5-40c9-900e-4f4320c075a0, 17, Finished, Available, Finished, False)

Total rows in Bronze: 1209

Top 10 by reading count:
+------------+--------------+---------+---------+---------+---------+--------+
|country_code|country_name  |parameter|avg_value|min_value|max_value|readings|
+------------+--------------+---------+---------+---------+---------+--------+
|NL          |Netherlands   |         |-145.57  |-999.0   |934.0    |117     |
|CL          |Chile         |         |140.67   |0.0      |2906.65  |81      |
|US          |United States |o3       |0.03     |0.01     |0.05     |72      |
|NL          |Netherlands   |pm10     |14.42    |6.46     |23.0     |48      |
|NL          |Netherlands   |no2      |7.91     |0.0      |19.2     |46      |
|IN          |India         |         |189.29   |0.02     |4300.0   |41      |
|NL          |Netherlands   |pm25     |7.6      |2.63     |16.6     |40      |
|MN          |Mongolia      |         |92.44    |2.0      |821.0    |39      |
|US          |United States |no2      |0.01     |0.0      |0.03     |38      |